In [ ]:
# install xgboost
#!pip install xgboost

# Subset for multi-output comparison

In [10]:
import anndata as ad

rna_hvg = ad.read_h5ad("../preprocessing/outputs/rna_hvg.h5ad")
pro_data = ad.read_h5ad("../preprocessing/outputs/protein_data.h5ad")

In [11]:
import numpy as np

n_bins = 10000

rng = np.random.RandomState(42)
n_bins = min(n_bins, rna_hvg.n_obs)
idx = rng.choice(rna_hvg.n_obs, size=n_bins, replace=False)

rna_sub = rna_hvg[idx].copy()
pro_sub = pro_data[idx].copy()

In [12]:
rna_sub.write_h5ad("subset_test/rna_sub.h5ad")
pro_sub.write_h5ad("subset_test/pro_sub.h5ad")

# Multi-output regression comparison run

In [13]:
from xgboost_multioutput import run_loop_benchmark

loop_output = run_loop_benchmark(
    rna_hvg_path="subset_test/rna_sub.h5ad",
    protein_path="subset_test/pro_sub.h5ad",
    cv_split_path=None,
    xgb_params=None,
    fold_to_test=0)

Fold 0: running per-protein loop (44 fits)...


/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:24] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:41] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:59] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:53:17] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserW

  loop time: 1784.1s, mean Pearson r: 0.3769


In [14]:
from xgboost_multioutput import run_multioutput_benchmark

mo_output = run_multioutput_benchmark(
    rna_hvg_path="subset_test/rna_sub.h5ad",
    protein_path="subset_test/pro_sub.h5ad",
    cv_split_path=None,
    xgb_params=None,
    fold_to_test=0,
    device="cpu")

Fold 0: running multi_output_tree on cpu (1 fit)...
  multi_output_tree time: 1838.3s, mean Pearson r: 0.3828


In [18]:
from xgboost_multioutput import compare_results

compared = compare_results(loop_output, mo_output).sort_values(by=["pearsonr_loop"], ascending=False)

Speedup: 1.0x
       protein  pearsonr_loop  pearsonr_multioutput  pearsonr_diff   r2_loop  \
33        PD-1       0.342089              0.310943      -0.031146  0.081656   
29        FIBR       0.469159              0.444178      -0.024982  0.180215   
7        CXCR5       0.441528              0.418765      -0.022763  0.180606   
13        CD44       0.460555              0.444937      -0.015618  0.212006   
23        SIRP       0.329141              0.315050      -0.014092  0.099287   
25        IDH1       0.345432              0.332042      -0.013390  0.105291   
26         MPO       0.607435              0.599495      -0.007940  0.354214   
4       CXCL13       0.308699              0.302055      -0.006644  0.075211   
36        MAP2       0.408707              0.404909      -0.003798  0.165428   
14         SMA       0.346526              0.343901      -0.002625  0.090550   
28        CD21       0.270338              0.267834      -0.002504  0.053566   
42        ICOS       0.305

In [19]:
compared

,protein,pearsonr_loop,pearsonr_multioutput,pearsonr_diff,r2_loop,r2_multioutput
26,MPO,0.607435,0.599495,-0.007940,0.354214,0.358631
12,CD68,0.558748,0.558645,-0.000104,0.295832,0.305996
32,TOX,0.545274,0.549272,0.003998,0.279334,0.292648
27,CD45,0.511952,0.525980,0.014028,0.240447,0.266569
35,CD4,0.497995,0.506344,0.008350,0.224124,0.237440
40,HLA-DR,0.492743,0.506597,0.013854,0.222349,0.244210
9,PD-L1,0.489263,0.493989,0.004727,0.223168,0.243666
31,CD3e,0.482122,0.505105,0.022982,0.195956,0.251330
22,CD74,0.476965,0.487666,0.010701,0.218661,0.233985
38,MGMT,0.470519,0.472043,0.001523,0.204969,0.220493
